In [1]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('bmh')
plt.rcParams['axes.facecolor'] = 'white'

In [2]:
y_true = np.load('../test-set-y_true.npy', allow_pickle=True)
y_pred_low = np.load('../test-set-y_pred_low.npy', allow_pickle=True)
y_pred_high = np.load('../test-set-y_pred_high.npy', allow_pickle=True)

In [13]:
mmsis = np.load('../all-mmsis-test.npy', allow_pickle=True)
days = np.load('../all-days-test.npy', allow_pickle=True)

## COG klaidos

In [6]:
mistake_mask = (
    (y_true[:, 0, 0] < y_pred_low[:, 0, 0]) |
    (y_true[:, 0, 0] > y_pred_high[:, 0, 0])
)

In [ ]:
mistake_counts = {}
mmsi_mistake_table = np.unique( mmsis[mistake_mask], return_counts=True )
days = np.unique( days[mistake_mask], return_counts=True )

for mmsi, mistake_count in zip(*mmsi_mistake_table):
    mistake_counts[mmsi] = mistake_count

In [8]:
mmsi_dict = {
    mmsi: mistakes
    for mmsi, mistakes in zip( *np.unique(mmsis, return_counts=True) )
}

In [9]:
import pandas as pd

In [10]:
mistake_df = {'mmsi': [], 'mistake_count': [], 'mistake_freq': [], 'total_seqs': []}

for mmsi, mistakes in mistake_counts.items():
    mistake_df['mmsi'].append(mmsi)
    mistake_df['mistake_count'].append(mistakes)
    mistake_df['total_seqs'].append(mmsi_dict[mmsi])
    mistake_df['mistake_freq'].append(mistakes / mmsi_dict[mmsi])

mistake_df = pd.DataFrame(mistake_df)

In [12]:
more_than_yipeng3 = mistake_df[ 1 - mistake_df['mistake_freq'] < 0.9 ]
more_than_yipeng3

,mmsi,mistake_count,mistake_freq,total_seqs
7,246447000,46,0.174905,263
9,255805899,38,0.209945,181
12,255807000,25,0.101215,247
15,305155000,36,0.129964,277
21,538011990,182,0.133725,1361
22,636018491,24,0.406780,59
25,636023244,176,0.129127,1363


In [19]:
df = pd.read_csv('../data/df-prepared.csv')

In [24]:
mmsi_mask = df['MMSI'].apply( lambda x: x in more_than_yipeng3['mmsi'].unique() )

In [26]:
df[mmsi_mask]['Cargo type'].unique()

<StringArray>
['No additional information', 'Category X', 'Reserved for future use']
Length: 3, dtype: str

In [29]:
df[mmsi_mask]['Length'].unique()

array([108., 149.,  89., 107., 109., 179., 172.])

In [30]:
df[mmsi_mask]['Width'].unique()

array([16., 23., 15., 27., 28.])

In [ ]:
thresholds = [0.865, 0.9, 0.95]

thresholds_false_positives = {}

for thresh in thresholds:
    false_positives = mistake_df[ 1 - mistake_df['mistake_freq'] < thresh ]
    print(thresh, false_positives.shape[0])

    thresholds_false_positives[thresh] = false_positives['mmsi'].unique()

0.865 13
0.9 18
0.95 28


## Laivų specifika (remiantis MarineTraffic ir VesselFinder)

219136000 - 200 x 36 m krovininis A pavojingumo klasės laivas, gabenantis labai pavojingą krovinį.

235102851 - 177 / 28 m krovininis laivas, gabenantis nepavojingą krovinį

244130689 - 196 / 26 m krovininis Ro-Ro A pavojingumo klasės laivas, gabenantis pavojingą krovinį.

244130873 - 137 / 22 m krovininis laivas, gabenantis nepavojingą krovinį

246191000 - 137 / 22 m krovininis A pavojingumo klasės laivas, gabenantis labai pavojingą krovinį.

255806507 - 300 / 48 m krovininis D pavojingumo klasės laivas.

257516000 - 213 / 32 m krovininis laivas

314837000 - 134 / 19 m krovininis laivas

341578001 - 89 / 13 m krovininis laivas

352586000 - 229 / 32 m

354711000 - 196 / 33 m

636016848 - 199 / 32 m

636017277 - 200 / 32 m

### Klaidų analizės planas:

1. Laivų specifikos (ilgis, plotis) prieš
2. Greičio ir pozicijos reikšmės
3. Požyimių grafikai
4. 